## **1st table: Average time to purchase by date** 

**The query was used to create a table which will show how much minutes it takes for each user to complete main steps for a purchase during the same day and it will show what operational system was used.** 


**Events selected:** 

- First `page_view_time` - Since we need to see when the user fist started the session 

- First `view_item` - we are interested when the user has started engaging by reviewing the items offered 

- First `add_to_cart` - After the user has revied items, we are interested when he has decided to purchase the item 

- Last `begin_checkout` - this step is needed to see when the user decided to move forward with his shopping experience, the user can move back and forward, so we need the last step here.  

- Last `add_payment_info` - To make a purchase, user needs to add his payment information, we are focusing on the completion, since user can go back and edit his cart.  

- Last `purchase` - The end result when the purshase is completed. 

**Query explanation:**

- First CTE filters users which have all mentioned steps, meaning we have full sessions. Also it transforms the timestamp to a date of event.

- Second CTE takes the first of each event type per user and day, but only for days with complete funnels by using JOIN on with same dates. 

- Main query will show the start time, which is the first visit and it will show the difference in minutes between each step. In the end it will show the end time which is the time of purchase. As a safety cause it will filter out all records which do not follow the sequence. 


In [ ]:
WITH complete_user_days AS (
  SELECT
    user_pseudo_id,
    EXTRACT(DATE FROM TIMESTAMP_MICROS(event_timestamp)) AS event_date
  FROM `tc-da-1.turing_data_analytics.raw_events`
  WHERE event_name IN ('page_view', 'view_item', 'add_to_cart', 'begin_checkout', 'add_payment_info', 'purchase')
  GROUP BY user_pseudo_id, EXTRACT(DATE FROM TIMESTAMP_MICROS(event_timestamp))
  HAVING 
    COUNT(DISTINCT CASE WHEN event_name = 'page_view' THEN event_name END) > 0 AND
    COUNT(DISTINCT CASE WHEN event_name = 'view_item' THEN event_name END) > 0 AND
    COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart' THEN event_name END) > 0 AND
    COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout' THEN event_name END) > 0 AND
    COUNT(DISTINCT CASE WHEN event_name = 'add_payment_info' THEN event_name END) > 0 AND
    COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN event_name END) > 0
),
first_events AS (
  SELECT 
    a.user_pseudo_id,
    a.event_date,
    MIN(CASE WHEN a.event_name = 'page_view' THEN TIMESTAMP_MICROS(a.event_timestamp) END) AS page_view_time,
    MIN(CASE WHEN a.event_name = 'view_item' THEN TIMESTAMP_MICROS(a.event_timestamp) END) AS view_item_time,
    MIN(CASE WHEN a.event_name = 'add_to_cart' THEN TIMESTAMP_MICROS(a.event_timestamp) END) AS add_to_cart_time,
    MAX(CASE WHEN a.event_name = 'begin_checkout' THEN TIMESTAMP_MICROS(a.event_timestamp) END) AS begin_checkout_time,
    MAX(CASE WHEN a.event_name = 'add_payment_info' THEN TIMESTAMP_MICROS(a.event_timestamp) END) AS add_payment_time,
    MAX(CASE WHEN a.event_name = 'purchase' THEN TIMESTAMP_MICROS(a.event_timestamp) END) AS purchase_time,
    a.operating_system
  FROM `tc-da-1.turing_data_analytics.raw_events` a
  JOIN complete_user_days c
    ON a.user_pseudo_id = c.user_pseudo_id 
    AND EXTRACT(DATE FROM TIMESTAMP_MICROS(a.event_timestamp)) = c.event_date
  GROUP BY a.user_pseudo_id, a.event_date, a.operating_system
)
SELECT
  user_pseudo_id,
  event_date,
  FORMAT_TIMESTAMP('%H:%M:%S', page_view_time) AS session_start,
  TIMESTAMP_DIFF(view_item_time, page_view_time, SECOND) AS seconds_to_view_item,
  TIMESTAMP_DIFF(add_to_cart_time, view_item_time, SECOND) AS seconds_to_add_to_cart,
  TIMESTAMP_DIFF(begin_checkout_time, add_to_cart_time, SECOND) AS seconds_to_begin_checkout,
  TIMESTAMP_DIFF(add_payment_time, begin_checkout_time, SECOND) AS seconds_to_add_payment,
  TIMESTAMP_DIFF(purchase_time, add_payment_time, SECOND) AS seconds_to_purchase,
  FORMAT_TIMESTAMP('%H:%M:%S', purchase_time) AS session_end,
  TIMESTAMP_DIFF(purchase_time, page_view_time, SECOND) AS total_session_seconds, 
  operating_system
FROM first_events
WHERE
  page_view_time <= view_item_time 
  AND view_item_time <= add_to_cart_time
  AND add_to_cart_time <= begin_checkout_time
  AND begin_checkout_time <= add_payment_time
  AND add_payment_time <= purchase_time
ORDER BY user_pseudo_id, event_date;

The results of the query can be found in Google sheets: [User main steps time](https://docs.google.com/spreadsheets/d/1AQ4hwDV-8RjMZyDgo3iN0U6lN2Nr_FZqa7LH-7VckhA/edit?usp=sharing)

## **2st table: A/B Testing for Same-day Conversion Times**   

**This query will output metrics which are needed for Evan's Awesome A/B Tools with 95% confidence:**  

`Mean:` The avg_seconds column gives the average time to purchase for each week  

`Std. Dev.:` The std_dev_seconds column provides the standard deviation  

`Count:` The user_count column gives the sample size    

[Evan's Awesome A/B Tool](https://www.evanmiller.org/ab-testing/t-test.html)

- To have same data as in first table, first CTE checks if the sessions contain same events and adds week number `Week 3` or `Week 4`.    

- Second CTE takes the first of each event type per user and day, but only for days with complete funnels by using JOIN on with same dates. 

- Main query calculates aggregated stats per week period and ensures both events exist and are in proper sequence

In [ ]:
WITH complete_user_days AS (
  SELECT
    user_pseudo_id,
    EXTRACT(DATE FROM TIMESTAMP_MICROS(event_timestamp)) AS event_date,
    CASE 
      WHEN DATE(TIMESTAMP_MICROS(event_timestamp)) BETWEEN DATE('2021-01-10') AND DATE('2021-01-16') THEN 'Week 3'
      WHEN DATE(TIMESTAMP_MICROS(event_timestamp)) BETWEEN DATE('2021-01-17') AND DATE('2021-01-23') THEN 'Week 4'
    END AS week_period
  FROM `tc-da-1.turing_data_analytics.raw_events`
  WHERE 
    event_name IN ('page_view', 'view_item', 'add_to_cart', 'begin_checkout', 'add_shipping_info', 'add_payment_info', 'purchase')
    AND DATE(TIMESTAMP_MICROS(event_timestamp)) BETWEEN DATE('2021-01-10') AND DATE('2021-01-23')
  GROUP BY user_pseudo_id, EXTRACT(DATE FROM TIMESTAMP_MICROS(event_timestamp)), 
    CASE 
      WHEN DATE(TIMESTAMP_MICROS(event_timestamp)) BETWEEN DATE('2021-01-10') AND DATE('2021-01-16') THEN 'Week 3'
      WHEN DATE(TIMESTAMP_MICROS(event_timestamp)) BETWEEN DATE('2021-01-17') AND DATE('2021-01-23') THEN 'Week 4'
    END
  HAVING 
    COUNT(DISTINCT CASE WHEN event_name = 'page_view' THEN event_name END) > 0 AND
    COUNT(DISTINCT CASE WHEN event_name = 'view_item' THEN event_name END) > 0 AND
    COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart' THEN event_name END) > 0 AND
    COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout' THEN event_name END) > 0 AND
    COUNT(DISTINCT CASE WHEN event_name = 'add_payment_info' THEN event_name END) > 0 AND
    COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN event_name END) > 0
),
session_durations AS (
  SELECT 
    c.user_pseudo_id,
    c.event_date,
    c.week_period,
    MIN(CASE WHEN a.event_name = 'page_view' THEN TIMESTAMP_MICROS(a.event_timestamp) END) AS page_view_time,
    MAX(CASE WHEN a.event_name = 'purchase' THEN TIMESTAMP_MICROS(a.event_timestamp) END) AS purchase_time,
    TIMESTAMP_DIFF(
      MAX(CASE WHEN a.event_name = 'purchase' THEN TIMESTAMP_MICROS(a.event_timestamp) END),
      MIN(CASE WHEN a.event_name = 'page_view' THEN TIMESTAMP_MICROS(a.event_timestamp) END),
      SECOND
    ) AS total_session_seconds
  FROM `tc-da-1.turing_data_analytics.raw_events` a
  JOIN complete_user_days c
    ON a.user_pseudo_id = c.user_pseudo_id 
    AND EXTRACT(DATE FROM TIMESTAMP_MICROS(a.event_timestamp)) = c.event_date
  WHERE 
    a.event_name IN ('page_view', 'purchase')
    AND DATE(TIMESTAMP_MICROS(a.event_timestamp)) BETWEEN DATE('2021-01-10') AND DATE('2021-01-23')
  GROUP BY c.user_pseudo_id, c.event_date, c.week_period
)
SELECT
  week_period,
  COUNT(DISTINCT user_pseudo_id) AS user_count,
  ROUND(AVG(total_session_seconds), 2) AS avg_seconds,
  ROUND(STDDEV(total_session_seconds), 2) AS std_dev_seconds
FROM session_durations
WHERE
  page_view_time IS NOT NULL
  AND purchase_time IS NOT NULL
  AND page_view_time < purchase_time
  AND week_period IS NOT NULL
GROUP BY week_period
ORDER BY week_period;

Query results: 

| week_period | user_count | avg_seconds | std_dev_seconds |
|-------------|------------|-------------|-----------------|
| Week 3      | 171        | 4041.01     | 8044.94         |
| Week 4      | 276        | 6670.48     | 13963.57        | 

![T-test results](https://i.imgur.com/IS3PbRI.png)

## **3rd table: User Funnel**  

 **The query was used to create a funnel table for the main steps across 4 weeks.** 

- Using the complete_user_days CTE similar to our previous approach to ensure we're analyzing the correct time periods.

- With unique_users_per_event we have unique users and make sure they had purchase during analysed period. 

- step_order is used to define the funnel step order for next query.

- Baseline count (page_view) for each week is created.

- Last CTE calculates the funnel conversion percentages. 

- Main query is pivoting the data to show all weeks as columns

In [ ]:
WITH complete_user_days AS (
  SELECT
    user_pseudo_id,
    EXTRACT(DATE FROM TIMESTAMP_MICROS(event_timestamp)) AS event_date,
    CASE 
      WHEN DATE(TIMESTAMP_MICROS(event_timestamp)) BETWEEN DATE('2021-01-10') AND DATE('2021-01-16') THEN 'Week 3'
      WHEN DATE(TIMESTAMP_MICROS(event_timestamp)) BETWEEN DATE('2021-01-17') AND DATE('2021-01-23') THEN 'Week 4'
      WHEN DATE(TIMESTAMP_MICROS(event_timestamp)) BETWEEN DATE('2021-01-24') AND DATE('2021-01-30') THEN 'Week 5'
    END AS week_period
  FROM `tc-da-1.turing_data_analytics.raw_events`
  WHERE 
    event_name IN ('page_view', 'view_item', 'add_to_cart', 'begin_checkout', 'add_shipping_info', 'add_payment_info', 'purchase')
    AND DATE(TIMESTAMP_MICROS(event_timestamp)) BETWEEN DATE('2021-01-10') AND DATE('2021-01-30')
  GROUP BY user_pseudo_id, EXTRACT(DATE FROM TIMESTAMP_MICROS(event_timestamp)), 
    CASE 
      WHEN DATE(TIMESTAMP_MICROS(event_timestamp)) BETWEEN DATE('2021-01-10') AND DATE('2021-01-16') THEN 'Week 3'
      WHEN DATE(TIMESTAMP_MICROS(event_timestamp)) BETWEEN DATE('2021-01-17') AND DATE('2021-01-23') THEN 'Week 4'
      WHEN DATE(TIMESTAMP_MICROS(event_timestamp)) BETWEEN DATE('2021-01-24') AND DATE('2021-01-30') THEN 'Week 5'
    END
),
unique_users_per_event AS (
  SELECT
    event_name,
    c.week_period,
    COUNT(DISTINCT a.user_pseudo_id) AS event_count
  FROM `tc-da-1.turing_data_analytics.raw_events` a
  JOIN complete_user_days c
    ON a.user_pseudo_id = c.user_pseudo_id 
    AND EXTRACT(DATE FROM TIMESTAMP_MICROS(a.event_timestamp)) = c.event_date
  WHERE 
    a.event_name IN ('page_view', 'view_item', 'add_to_cart', 'begin_checkout', 'add_shipping_info', 'add_payment_info', 'purchase')
    AND DATE(TIMESTAMP_MICROS(a.event_timestamp)) BETWEEN DATE('2021-01-10') AND DATE('2021-01-30')
  GROUP BY event_name, c.week_period
),
step_order AS (
  SELECT 'page_view' AS event_name, 1 AS event_order UNION ALL
  SELECT 'view_item', 2 UNION ALL
  SELECT 'add_to_cart', 3 UNION ALL
  SELECT 'begin_checkout', 4 UNION ALL
  SELECT 'add_shipping_info', 5 UNION ALL
  SELECT 'add_payment_info', 6 UNION ALL
  SELECT 'purchase', 7
),
base_counts AS (
  SELECT
    week_period,
    event_count AS base_count
  FROM unique_users_per_event
  WHERE event_name = 'page_view'
),
calculated_funnel AS (
  SELECT
    u.event_name,
    s.event_order,
    u.week_period,
    u.event_count,
    b.base_count,
    ROUND((u.event_count / b.base_count) * 100, 2) AS pct_from_start
  FROM unique_users_per_event u
  JOIN step_order s ON u.event_name = s.event_name
  JOIN base_counts b ON u.week_period = b.week_period
)
SELECT
  event_name,
  event_order,
  MAX(CASE WHEN week_period = 'Week 3' THEN event_count ELSE 0 END) AS week_3_count,
  MAX(CASE WHEN week_period = 'Week 4' THEN event_count ELSE 0 END) AS week_4_count,
  MAX(CASE WHEN week_period = 'Week 5' THEN event_count ELSE 0 END) AS week_5_count,
  MAX(CASE WHEN week_period = 'Week 3' THEN pct_from_start ELSE 0 END) AS week_3_pct,
  MAX(CASE WHEN week_period = 'Week 4' THEN pct_from_start ELSE 0 END) AS week_4_pct,
  MAX(CASE WHEN week_period = 'Week 5' THEN pct_from_start ELSE 0 END) AS week_5_pct
FROM calculated_funnel
GROUP BY event_name, event_order
ORDER BY event_order;

Query results: 

|    event_name     | event_order | week_3_count | week_4_count | week_5_count | week_3_pct | week_4_pct | week_5_pct |
|-------------------|-------------|--------------|--------------|--------------|------------|------------|------------|
|     page_view     |      1      |     23267    |     22563    |     21576    |    100.0   |    100.0   |    100.0   |
|     view_item     |      2      |      4370    |      5565    |      5643    |    18.78   |    24.66   |    26.15   |
|    add_to_cart    |      3      |       985    |      1022    |       995    |     4.23   |     4.53   |     4.61   |
| begin_checkout    |      4      |       428    |       586    |       555    |     1.84   |     2.59   |     2.57   |
| add_payment_info  |      5      |       308    |       449    |       415    |     1.32   |     1.99   |     1.92   |
|     purchase      |      6      |       224    |       346    |       324    |     0.96   |     1.53   |     1.50   |

## **4th table: Churned used time**  

 **The query was used to create a time table of events for churned customers.** 

- First creates a CTE called churned_users that identifies users who had events after January 4, 2021, including page views, item views, and cart additions but no purchases
- Second CTE called first_events finds the earliest timestamp for each of those event types per user per day
- Final CTE calculates the time differences between these events in seconds.

In [ ]:
  WITH churned_users AS (
  SELECT
    user_pseudo_id,
    EXTRACT(DATE FROM TIMESTAMP_MICROS(event_timestamp)) AS event_date
  FROM `tc-da-1.turing_data_analytics.raw_events`
WHERE EXTRACT(DATE FROM TIMESTAMP_MICROS(event_timestamp)) > '2021-01-04'
  AND event_name IN ('page_view', 'view_item', 'add_to_cart')
  AND event_name != 'purchase'
  GROUP BY user_pseudo_id, EXTRACT(DATE FROM TIMESTAMP_MICROS(event_timestamp))
),
first_events AS (
  SELECT 
    a.user_pseudo_id,
    c.event_date,
    MIN(CASE WHEN a.event_name = 'page_view' THEN TIMESTAMP_MICROS(a.event_timestamp) END) AS page_view_time,
    MIN(CASE WHEN a.event_name = 'view_item' THEN TIMESTAMP_MICROS(a.event_timestamp) END) AS view_item_time,
    MIN(CASE WHEN a.event_name = 'add_to_cart' THEN TIMESTAMP_MICROS(a.event_timestamp) END) AS add_to_cart_time,
    FROM `tc-da-1.turing_data_analytics.raw_events` a
  JOIN churned_users c
    ON a.user_pseudo_id = c.user_pseudo_id 
    AND EXTRACT(DATE FROM TIMESTAMP_MICROS(a.event_timestamp)) = c.event_date
  GROUP BY a.user_pseudo_id, c.event_date
)
SELECT
  user_pseudo_id,
  event_date,
  TIMESTAMP_DIFF(view_item_time, page_view_time, SECOND) AS seconds_to_view_item,
  TIMESTAMP_DIFF(add_to_cart_time, view_item_time, SECOND) AS seconds_to_add_to_cart
FROM first_events
WHERE
  page_view_time <= view_item_time 
  AND view_item_time <= add_to_cart_time
ORDER BY user_pseudo_id, event_date;

The results of the query can be found in Google sheets: [Churned customers data ](https://docs.google.com/spreadsheets/d/1y3IorJmDcyYxmlFNbja-egybj6m_aOTgJHkmno2aA6g/edit?usp=sharing)
